## Project Gutenberg  
陳柏謙 Andy Chen

In [100]:
# 使用 requests 工具
import requests

# 使用 json 工具
import json

# 匯入 bs4 套件
from bs4 import BeautifulSoup as bs

# from selenium.webdriver.chrome.service import Service
from selenium import webdriver

# import regex
import re

# import os for filesafe
import os

# pprint
from pprint import pprint

In [48]:
# Address of Gutenberg zh
url = "https://www.gutenberg.org/browse/languages/zh"

# Use GET to download the web
res = requests.get(url)

# 指定 lxml 作為解析器
soup = bs(res.text, "lxml")

利用 Dictionary key值不可重複的特性  
將書本資訊依照{書本標題：編號}存取  
存取出不重複，共355本中文書籍清單

In [99]:
# 建立 Dict 來放置列表資訊
books_dicts = {}
books_dicts.clear()

regex_full_Chinese = r"[\u4E00-\u9FFF\u3000-\u303F\uFF00-\uFFEF]+"

for title_tag in soup.select("li.pgdbetext > a"):
    title_text = title_tag.get_text()                   # book title
    href_text = title_tag.get("href")                   # book href /ebooks/25328
    get_num = re.search(r"/ebooks/(\d+)", href_text)

    if re.fullmatch(regex_full_Chinese, title_text):
        # list_books.append(title_text)
        book_num = get_num.group(1)
        books_dicts[title_text] = book_num
        # books_dicts[book_num] = title_text
        # print(title_text)
    else:
        pass
print(books_dicts)
print(f"共{len(books_dicts)}本")


{'豆棚閒話': '25328', '戲中戲': '24225', '比目魚': '27119', '三字經': '25160', '山水情': '25146', '山海經': '25288', '施公案': '25393', '易經': '25501', '木蘭奇女傳': '23938', '海公案': '54494', '燕丹子': '24068', '狄公案': '27686', '百家姓': '25196', '禮記': '24048', '綠牡丹': '27330', '詩經': '23873', '麟兒報': '27399', '天豹圖': '26904', '梁公九諫': '26886', '長恨歌': '25352', '李娃傳': '24051', '玉樓春': '25422', '漢書': '23841', '引鳳蕭': '26921', '今古奇觀': '24230', '後西游記': '27332', '飛跎全傳': '27331', '佛說四十二章經': '23585', '紅樓夢': '24264', '洛神賦': '24041', '水滸後傳': '25217', '幼學瓊林': '52269', '治世餘聞': '26932', '琵琶記': '25246', '雪月梅傳': '26739', '龍川詞': '26873', '三國志': '25606', '隋唐演義': '23835', '論語': '23839', '白圭志': '27023', '孟子字義疏證': '25360', '安樂集': '24106', '鄧析子': '7215', '醉醒石': '24027', '唐鍾馗平鬼傳': '27329', '春秋繁露': '25385', '虬髯客傳': '23915', '吳船錄': '27581', '星槎勝覽': '23982', '喻世明言': '27582', '平妖傳': '25248', '東周列國志': '25349', '警世通言': '24141', '醒世恆言': '24239', '封氏聞見記': '27207', '搜神記': '25362', '抱朴子': '25696', '西京雜記': '23951', '幽明錄': '52278', '明鏡公案': '52280', '公孫龍子': '72

In [ ]:
# books_dicts = {
#     "豆棚閒話" : "25328",
#     "傳習錄" : "25517"
#     "周髀算經" : "12408"
# }

output_dir = "project_gutenberg"
# output_dir1 = "test1"

list_content = []

regex_chinese = re.compile(r"[\u4e00-\u9fff]")
regex_english = re.compile(r"[A-Za-z]")
# regex_chinese2 = re.compile(r"^[\u4e00-\u9fff\s，。！？「」『』、．：；《》【】（）—…]*$")

for title, book_num in books_dicts.items():
    url_content = f"https://www.gutenberg.org/cache/epub/{book_num}/pg{book_num}-images.html"
    list_content.clear()

    try:
        res_content = requests.get(url_content)
        soup_content = bs(res_content.text, "lxml")


        # ["p", "div"]
        for p in soup_content.find_all("p"):
            raw_text = p.get_text()
            # print(raw_text)
            clean_text = raw_text.replace("\xa0", " ").replace("\r\n", " ").replace("\n", " ").replace('&nbsp;', ' ').strip()
            # print(clean_text)

            # if regex_english.search(clean_text):
            #     print(clean_text)
                # list_content.append(clean_text)

            # if regex_chinese2.search(clean_text):
            #     print(clean_text)
            #     list_content.append(clean_text)

            if not regex_english.search(clean_text):  # Not include English
                print(clean_text)
                list_content.append(clean_text)

        
        # safe as txt file
        output_path = os.path.join(output_dir, f"{title}.txt")
        with open(output_path, "w", encoding="utf-8") as f:
            for p in list_content:
                f.write(p + "\n\n")
        print(f"Aready Safed: {title}.txt")

        

    except Exception as e:
        print(f"error, cannot get the book {title}: {e}")
